# PySpark CDC — SCD Type 2 History

Historized CDC example. Updates close the current row and insert a new version. Deletes close the current row only.

## 00 — Spark setup and sample data

In [1]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import *

spark = SparkSession.getActiveSession()
if spark is not None:
    spark.stop()

spark = (
    SparkSession.builder
    .appName("cdc-scd2-history")
    .master("spark://spark-master:7077")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.warehouse.dir", "/tmp/cdc-scd2-history_warehouse")
    .getOrCreate()
)

spark.version

'4.0.2'

In [2]:
customers_history_schema = StructType([
    StructField("customer_id", IntegerType(), False),
    StructField("name", StringType(), True),
    StructField("email", StringType(), True),
    StructField("valid_from", StringType(), False),
    StructField("valid_to", StringType(), True),
    StructField("is_current", BooleanType(), False),
])

cdc_events_schema = StructType([
    StructField("customer_id", IntegerType(), False),
    StructField("name", StringType(), True),
    StructField("email", StringType(), True),
    StructField("op", StringType(), False),
    StructField("event_time", StringType(), False),
])

customers_history = spark.createDataFrame([
    (1, "Alice", "alice@old.com", "2026-05-02T09:00:00", None, True),
    (2, "Bob", "bob@mail.com", "2026-05-02T09:00:00", None, True),
    (3, "Carol", "carol@mail.com", "2026-05-02T09:00:00", None, True),
], customers_history_schema) \
    .withColumn("valid_from", F.to_timestamp("valid_from")) \
    .withColumn("valid_to", F.to_timestamp("valid_to"))

cdc_events = spark.createDataFrame([
    (1, "Alice", "alice@new.com", "U", "2026-05-02T10:05:00"),
    (2, None, None, "D", "2026-05-02T10:10:00"),
    (4, "Diana", "diana@mail.com", "I", "2026-05-02T10:15:00"),
    (1, "Alice A.", "alice.a@mail.com", "U", "2026-05-02T10:20:00"),
], cdc_events_schema).withColumn(
    "event_time",
    F.to_timestamp("event_time")
)

print("History before CDC")
customers_history.orderBy("customer_id", "valid_from").show(truncate=False)

print("CDC events")
cdc_events.orderBy("event_time").show(truncate=False)

History before CDC


+-----------+-----+--------------+-------------------+--------+----------+
|customer_id|name |email         |valid_from         |valid_to|is_current|
+-----------+-----+--------------+-------------------+--------+----------+
|1          |Alice|alice@old.com |2026-05-02 09:00:00|NULL    |true      |
|2          |Bob  |bob@mail.com  |2026-05-02 09:00:00|NULL    |true      |
|3          |Carol|carol@mail.com|2026-05-02 09:00:00|NULL    |true      |
+-----------+-----+--------------+-------------------+--------+----------+

CDC events
+-----------+--------+----------------+---+-------------------+
|customer_id|name    |email           |op |event_time         |
+-----------+--------+----------------+---+-------------------+
|1          |Alice   |alice@new.com   |U  |2026-05-02 10:05:00|
|2          |NULL    |NULL            |D  |2026-05-02 10:10:00|
|4          |Diana   |diana@mail.com  |I  |2026-05-02 10:15:00|
|1          |Alice A.|alice.a@mail.com|U  |2026-05-02 10:20:00|
+-----------+--

## 01 — Latest event per key

In [3]:
from pyspark.sql.window import Window

latest_event_window = Window.partitionBy("customer_id").orderBy(F.col("event_time").desc())

latest_cdc = (
    cdc_events
    .withColumn("rn", F.row_number().over(latest_event_window))
    .filter(F.col("rn") == 1)
    .drop("rn")
)

latest_cdc.orderBy("customer_id").show(truncate=False)

+-----------+--------+----------------+---+-------------------+
|customer_id|name    |email           |op |event_time         |
+-----------+--------+----------------+---+-------------------+
|1          |Alice A.|alice.a@mail.com|U  |2026-05-02 10:20:00|
|2          |NULL    |NULL            |D  |2026-05-02 10:10:00|
|4          |Diana   |diana@mail.com  |I  |2026-05-02 10:15:00|
+-----------+--------+----------------+---+-------------------+



## 02 — Close current records

Updates and deletes close the currently active historical record.

In [4]:
events_to_close = (
    latest_cdc
    .filter(F.col("op").isin("U", "D"))
    .select("customer_id", F.col("event_time").alias("close_time"))
)

closed_history = (
    customers_history.alias("h")
    .join(events_to_close.alias("e"), on="customer_id", how="left")
    .select(
        "customer_id",
        "name",
        "email",
        "valid_from",
        F.when(
            (F.col("h.is_current") == True) & F.col("e.close_time").isNotNull(),
            F.col("e.close_time")
        ).otherwise(F.col("h.valid_to")).alias("valid_to"),
        F.when(
            (F.col("h.is_current") == True) & F.col("e.close_time").isNotNull(),
            F.lit(False)
        ).otherwise(F.col("h.is_current")).alias("is_current")
    )
)

closed_history.orderBy("customer_id", "valid_from").show(truncate=False)

+-----------+-----+--------------+-------------------+-------------------+----------+
|customer_id|name |email         |valid_from         |valid_to           |is_current|
+-----------+-----+--------------+-------------------+-------------------+----------+
|1          |Alice|alice@old.com |2026-05-02 09:00:00|2026-05-02 10:20:00|false     |
|2          |Bob  |bob@mail.com  |2026-05-02 09:00:00|2026-05-02 10:10:00|false     |
|3          |Carol|carol@mail.com|2026-05-02 09:00:00|NULL               |true      |
+-----------+-----+--------------+-------------------+-------------------+----------+



## 03 — Insert new versions

Inserts and updates create a new current version. Deletes do not create a replacement row.

In [5]:
new_versions = (
    latest_cdc
    .filter(F.col("op").isin("I", "U"))
    .select(
        "customer_id",
        "name",
        "email",
        F.col("event_time").alias("valid_from"),
        F.lit(None).cast("timestamp").alias("valid_to"),
        F.lit(True).alias("is_current")
    )
)

history_after_cdc = closed_history.unionByName(new_versions)

history_after_cdc.orderBy("customer_id", "valid_from").show(truncate=False)

+-----------+--------+----------------+-------------------+-------------------+----------+
|customer_id|name    |email           |valid_from         |valid_to           |is_current|
+-----------+--------+----------------+-------------------+-------------------+----------+
|1          |Alice   |alice@old.com   |2026-05-02 09:00:00|2026-05-02 10:20:00|false     |
|1          |Alice A.|alice.a@mail.com|2026-05-02 10:20:00|NULL               |true      |
|2          |Bob     |bob@mail.com    |2026-05-02 09:00:00|2026-05-02 10:10:00|false     |
|3          |Carol   |carol@mail.com  |2026-05-02 09:00:00|NULL               |true      |
|4          |Diana   |diana@mail.com  |2026-05-02 10:15:00|NULL               |true      |
+-----------+--------+----------------+-------------------+-------------------+----------+



## 04 — Current records from SCD2

In [6]:
history_after_cdc.filter(F.col("is_current") == True).orderBy("customer_id").show(truncate=False)

+-----------+--------+----------------+-------------------+--------+----------+
|customer_id|name    |email           |valid_from         |valid_to|is_current|
+-----------+--------+----------------+-------------------+--------+----------+
|1          |Alice A.|alice.a@mail.com|2026-05-02 10:20:00|NULL    |true      |
|3          |Carol   |carol@mail.com  |2026-05-02 09:00:00|NULL    |true      |
|4          |Diana   |diana@mail.com  |2026-05-02 10:15:00|NULL    |true      |
+-----------+--------+----------------+-------------------+--------+----------+

